<a href="https://colab.research.google.com/github/caiyunkai404602-jpg/DSAA2011-TaskA-RainAustralia/blob/main/task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ==========================================
# Task 1: Data Preprocessing
# ==========================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Load data and initial exploration
# Make sure 'data/weatherAUS.csv' exists in your Colab environment
df = pd.read_csv('data/weatherAUS.csv')
print(f"Original dataset shape: {df.shape}")

# Calculate missing value percentages (useful for report insights)
missing_ratio = (df.isnull().sum() / len(df)) * 100
missing_ratio = missing_ratio[missing_ratio > 0].sort_values(ascending=False)
print("\nTop missing value percentages:")
print(missing_ratio.head())

# ==========================================
# 1.1 Examine and Handle Missing Values
# ==========================================
# (1) Drop rows where the target variable 'RainTomorrow' is missing
df_clean = df.dropna(subset=['RainTomorrow']).copy()
print(f"\nShape after dropping missing target values: {df_clean.shape}")

# (2) Add missing indicators for features with high missing rates (>35%)
high_missing_cols = ['Sunshine', 'Evaporation', 'Cloud9am', 'Cloud3pm']
for col in high_missing_cols:
    if col in df_clean.columns:
        df_clean[f'{col}_is_missing'] = df_clean[col].isnull().astype(int)

# Separate numeric and categorical columns
num_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

# (3) Impute numeric features with median to resist outliers
for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# (4) Impute categorical features with mode
for col in cat_cols:
    if col not in ['Date', 'RainTomorrow']:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(f"Total missing values remaining after imputation: {df_clean.isnull().sum().sum()}")

# ==========================================
# 1.2 Handle Non-numeric Values
# ==========================================
# (1) Extract 'Month' from 'Date' to preserve seasonality, then drop 'Date'
if 'Date' in df_clean.columns:
    df_clean['Date'] = pd.to_datetime(df_clean['Date'])
    df_clean['Month'] = df_clean['Date'].dt.month
    df_clean.drop(columns=['Date'], inplace=True)

# (2) Boolean mapping for binary features
df_clean['RainToday'] = df_clean['RainToday'].map({'No': 0, 'Yes': 1})
df_clean['RainTomorrow'] = df_clean['RainTomorrow'].map({'No': 0, 'Yes': 1})

# (3) One-Hot Encoding for multi-categorical features
multi_cat_cols = [c for c in ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm'] if c in df_clean.columns]
df_encoded = pd.get_dummies(df_clean, columns=multi_cat_cols, drop_first=True, dtype=int)

print(f"Feature matrix shape after encoding: {df_encoded.shape}")

# ==========================================
# 1.3 Further Processing: Standardization
# ==========================================
# Separate features X and target y
X = df_encoded.drop(columns=['RainTomorrow'])
y = df_encoded['RainTomorrow']

# Standardize continuous numeric features only (exclude dummy variables)
continuous_cols = [c for c in num_cols if c in X.columns and not c.endswith('_is_missing')] + ['Month']
scaler = StandardScaler()

X_scaled = X.copy()
X_scaled[continuous_cols] = scaler.fit_transform(X[continuous_cols])

print(f"\nData preprocessing completed!")
print(f"Final scaled feature matrix shape: {X_scaled.shape}")

Original dataset shape: (145460, 23)

Top missing value percentages:
Sunshine       48.009762
Evaporation    43.166506
Cloud3pm       40.807095
Cloud9am       38.421559
Pressure9am    10.356799
dtype: float64

Shape after dropping missing target values: (142193, 23)
Total missing values remaining after imputation: 0
Feature matrix shape after encoding: (142193, 116)

Data preprocessing completed!
Final scaled feature matrix shape: (142193, 115)


In [6]:
# Save the cleaned features X_scaled and target y as separate CSV files
X_scaled.to_csv('data/X_scaled_cleaned.csv', index=False)
y.to_csv('data/y_cleaned.csv', index=False)
print("Cleaned data has been successfully saved to the data folder!")

Cleaned data has been successfully saved to the data folder!
